In [ ]:
import math

def count_tokens(df: pd.DataFrame, column: str) -> Counter[str]:
    """Count token frequencies in a nested token column.

    Args:
        df: Input dataframe with tokenized documents.
        column: Column name that stores nested sentence tokens.

    Returns:
        Token frequency counter.
    """
    counter: Counter[str] = Counter()
    for doc in tqdm(df[column], desc="Counting Tokens", leave=False):
        for sentence in doc:
            for token in sentence:
                counter[token] += 1
    return counter

class KeynessComparer:
    """Compare token salience between two corpora."""

    def __init__(
        self,
        df1: pd.DataFrame,
        df2: pd.DataFrame,
        column: str = "lemmas",
        freq_filter: int = 5,
    ) -> None:
        """Prepare counters and contingency table for keyness metrics.

        Args:
            df1: First corpus dataframe.
            df2: Second corpus dataframe.
            column: Token column used for counting.
            freq_filter: Minimum frequency to keep a token.
        """
        self.df1 = df1
        self.df2 = df2
        self.column = column
        self.freq_filter = freq_filter

        self.counter1 = count_tokens(self.df1, self.column)
        self.counter2 = count_tokens(self.df2, self.column)

        self.total_tokens1 = sum(self.counter1.values())
        self.total_tokens2 = sum(self.counter2.values())

        # Apply the frequency filter
        self.counter1 = Counter({token: count for token, count in self.counter1.items() if count >= self.freq_filter})
        self.counter2 = Counter({token: count for token, count in self.counter2.items() if count >= self.freq_filter})

        self.df = self.build_contingency_df()

    def build_contingency_df(self) -> pd.DataFrame:
        """Build observed and expected counts for each token."""
        all_tokens = set(self.counter1.keys()) | set(self.counter2.keys())
        data = []
        for token in tqdm(all_tokens, desc="Build Contingency Table"):
            observed_tok_1 = self.counter1[token]
            observed_tok_2 = self.counter2[token]
            observed_notok_1 = self.total_tokens1 - observed_tok_1
            observed_notok_2 = self.total_tokens2 - observed_tok_2

            # Expected frequencies under the null hypothesis of no association between token and corpus
            expected_tok_1 = (observed_tok_1 + observed_tok_2) * self.total_tokens1 / (self.total_tokens1 + self.total_tokens2)
            expected_tok_2 = (observed_tok_1 + observed_tok_2) * self.total_tokens2 / (self.total_tokens1 + self.total_tokens2)
            expected_notok_1 = (observed_notok_1 + observed_notok_2) * self.total_tokens1 / (self.total_tokens1 + self.total_tokens2)
            expected_notok_2 = (observed_notok_1 + observed_notok_2) * self.total_tokens2 / (self.total_tokens1 + self.total_tokens2)

            data.append({
                "token": token,
                "observed_tok_1": observed_tok_1,
                "observed_tok_2": observed_tok_2,
                "observed_notok_1": observed_notok_1,
                "observed_notok_2": observed_notok_2,
                "expected_tok_1": expected_tok_1,
                "expected_tok_2": expected_tok_2,
                "expected_notok_1": expected_notok_1,
                "expected_notok_2": expected_notok_2,
                "prevalent_1": True if observed_tok_1 > expected_tok_1 else False,
            })
        contingency_df = pd.DataFrame(data)
        return contingency_df

    def log_likelihood_ratio(self, contingency_row: pd.Series) -> float:
        """Compute log-likelihood ratio for one contingency row."""
        g_value = 0.0
        for token_presence in ["tok", "notok"]:
            for corpus in ["1", "2"]:
                observed = contingency_row[f"observed_{token_presence}_{corpus}"]
                expected = contingency_row[f"expected_{token_presence}_{corpus}"]

                if observed > 0 and expected > 0:
                    g_value += observed * math.log(observed / expected)
        return 2 * g_value

    def log_ratio(self, contingency_row: pd.Series) -> float:
        """Compute smoothed log-ratio for one contingency row."""
        observed_tok_1 = contingency_row["observed_tok_1"]
        observed_tok_2 = contingency_row["observed_tok_2"]
        logprob1 = math.log((observed_tok_1 + 1/2) / (self.total_tokens1 + 1/2))  # Add-one smoothing
        logprob2 = math.log((observed_tok_2 + 1/2) / (self.total_tokens2 + 1/2))  # Add-one smoothing

        return logprob1 - logprob2

    def build_keyness_df(self) -> pd.DataFrame:
        """Attach LLR and log-ratio columns to the contingency dataframe."""
        tqdm.pandas(desc="Calculating log likelihood ratio")
        self.df["llr"] = self.df.progress_apply(self.log_likelihood_ratio, axis=1)
        tqdm.pandas(desc="Calculating log ratio")
        self.df["log_ratio"] = self.df.progress_apply(self.log_ratio, axis=1)
        return self.df

In [ ]:
df_pre1920 = raw_df[raw_df["year"] < 1920]
df_post1920 = raw_df[raw_df["year"] >= 1920]

kf = KeynessComparer(df_pre1920, df_post1920)

In [ ]:
df_pre1920 = raw_df[raw_df["year"] < 1920]

df_tat = df_pre1920[df_pre1920["meta.mediaTitle"] == "Die Tat"]
df_nzz = df_pre1920[df_pre1920["meta.mediaTitle"] == "Neue Zürcher Zeitung"]
kf = KeynessComparer(df_tat, df_nzz, freq_filter=3)

In [ ]:
kf.__class__ = KeynessComparer

In [ ]:
kf.build_keyness_df()

In [ ]:
kf.df.sort_values("log_ratio", ascending=False).head(10)